In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from sklearn.metrics import mean_absolute_error, mean_squared_error
from models import *
from plots import *
from statsmodels.stats.diagnostic import acorr_ljungbox
import pandas as pd
import itertools

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = load_data()

In [ ]:
data.head()

In [ ]:
target = 't_seasdiff'
exog=['lagged_tmed_24', 'lagged_prec_24', 'lagged_tmin_24', 'lagged_tmax_24']

In [ ]:
data.head()

In [ ]:
if target == 't_seasdiff':
    data["t_seasdiff"]= data["tdiff"].diff(24)
    data = data.dropna(subset=["t_seasdiff"])

# SARIMA / SARIMAX

In [ ]:
len(data)

In [ ]:
p, d, q = 0, 1, 1
P, D, Q, s = 1, 0, 0, 12


In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s, exog= exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
from utils import sarima_candidates_from_acf

cand_res = sarima_candidates_from_acf(data[target], m=12, max_p=3, max_q=2, max_P=3, max_Q=3, d=0, D=0 ,max_lag=50)

candidates = cand_res["candidates"]
for i, (p, d, q, P, D, Q, s) in enumerate(candidates, 1):
    print(f"{i:02d}. SARIMA({p},{d},{q})({P},{D},{Q},{s})")

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=train,
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=exog,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

## Using AutoSarima with the current train/test split

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12, exog=exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
best_info, scores_df = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=None,
    metric="rmse",
    use="avg",
    verbose=True,
)
cv = best_info["cv_results"]

In [ ]:
plot_sarima_cv_results(
    df=data,
    target=target,
    split_metrics=cv.get("split_metrics", []),  
    forecasts=cv.get("forecasts", []),
    actuals=cv.get("actuals", []),
    forecast_months=24,
    residuals=cv.get("residuals", []),
    lb=pd.DataFrame(cv.get("lb_results", [])),
)

In [ ]:
last_fold= cv["folds"][-1]
print(last_fold["y_pred"].index)
print(last_fold["y_true"].index)
print(np.var(last_fold["y_pred"]))
print(np.var(last_fold["y_true"]))

In [ ]:
plot_last_fold(last_fold, acf_lags=12, lb_lags=1)

In [ ]:
best_info, scores_df = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    train_years=8,
    forecast_months=24,
    test_start_year=1995,
    exog=exog,
    metric="rmse",
    use="avg",
    verbose=True,
)
cv = best_info["cv_results"]

In [ ]:
plot_sarima_cv_results(
    df=data,
    target=target,
    split_metrics=cv.get("split_metrics", []),  
    forecasts=cv.get("forecasts", []),
    actuals=cv.get("actuals", []),
    forecast_months=24,
    residuals=cv.get("residuals", []),
    lb=pd.DataFrame(cv.get("lb_results", [])),
)

In [ ]:
last_fold= cv["folds"][-1]
plot_last_fold(last_fold, acf_lags=12, lb_lags=12)

In [ ]:
last_fold

# LSTMs

In [ ]:
res = lstm_grid_search_cv(data, 'tdiff', train_years=8, forecast_months=24, test_start_year=1996, epochs=60, verbose=0, param_grid={
    'exog': [['lagged_tmed_24', 'lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'], ['lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'], ['lagged_tmed_24', 'lagged_prec_24'], ['lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24']],
    'lr': [0.001],
    'hidden_size': [50, 25, 100],
    'num_layers': [1, 2],
    'dropout': [0.1, 0.2, 0.3]
    
})

In [ ]:
print_grid_search_results(res['results'], res['best_overall_score'], res['best_overall_results'], res['best_overall_params'],res['best_avg_score'], res['best_avg_results'], res['best_avg_params'],res['best_last_fold_score'], res['best_last_fold_results'], res['best_last_fold_params'])

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmed_24', 'lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'],  
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=100,
    num_layers=2,
    dropout=0.3,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=2010,
    exog=['lagged_tmed_24', 'lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'],  
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=100,
    num_layers=2,
    dropout=0.3,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog= ['lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'], 
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=50,
    num_layers=2,
    dropout=0.3,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=2010,
    exog= ['lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24'], 
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=50,
    num_layers=2,
    dropout=0.3,
    plot=True,
)

# Prophet

Prophet is an open-source forecasting model developed by Facebook (Meta) designed specifically for time series forecasting. It focuses on providing accurate forecasts with minimal manual tuning and is well-suited for business-related time series such as sales, demand, traffic, and activity metrics.

In [ ]:
def evaluate(y_test, y_pred):
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
loaded_data = load_data()
data = pd.DataFrame({
    'date': loaded_data['date'],
    'y': loaded_data['tdiff']
})

data.rename(columns={'date': 'ds', 'value': 'y'}, inplace=True)

data

In [ ]:
# Use first 80% for training, last 20% for testing
split_idx = int(len(data)*0.8)
train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

X_train = train.drop(columns=['y'])
y_train = train['y']
X_test = test.drop(columns=['y'])
y_test = test['y']

print(len(test))
print(train.dtypes)
y_test

## Hyperparameter Tuning
In this section, we tune Prophet’s hyperparameters to identify the parameter combination that yields the best performance for our dataset. To achieve this, we use Prophet’s built-in diagnostic tools, **cross_validation** and **performance_metrics**.

The hyperparameters that we tune are:

* **changepoint_prior_scale**: This is probably the most impactful parameter. It determines the flexibility of the trend, and in particular how much the trend changes at the trend changepoints. If it is too small, the trend will be underfit and variance that should have been modeled with trend changes will instead end up being handled with the noise term. If it is too large, the trend will overfit and in the most extreme case the trend will capture yearly seasonality;

* **seasonality_prior_scale**: This parameter controls the flexibility of the seasonality. Similarly, a large value allows the seasonality to fit large fluctuations, a small value shrinks the magnitude of the seasonality;

* **seasonality_mode**: This parameters determines if the model should model an additive or multiplicative seasonality. Options are "additive" or "multiplicative";

In [ ]:
from prophet.diagnostics import performance_metrics
from prophet.diagnostics import cross_validation
import itertools
from prophet import Prophet

# Define the parameter grid
param_grid = {  
    'changepoint_prior_scale': [0.001, 0.01, 0.1, 0.5],
    'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0],
    "seasonality_mode": ['additive', 'multiplicative']
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here

# Use cross validation to evaluate all parameters
for params in all_params:
    m = Prophet(**params).fit(data)  # Fit model with given params
    df_cv = cross_validation(m, initial="7304 days", horizon='1826 days', period= "30 days", parallel="processes")
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses
best_params = tuning_results.sort_values(by='rmse').iloc[0].to_dict()
print(best_params)


In [ ]:
# Fit the model with the best parameters
model = Prophet(changepoint_prior_scale=best_params['changepoint_prior_scale'],
                seasonality_prior_scale=best_params['seasonality_prior_scale'],
                seasonality_mode=best_params['seasonality_mode'])
model.fit(train)
future = model.make_future_dataframe(periods=len(test), freq='MS')

forecast = model.predict(future)

# Filter forecast to only the test dates
forecast_test = forecast.set_index('ds').loc[y_test.index]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test, label="Actual", linewidth=2)
plt.plot(y_test.index, forecast_test['yhat'], label="Predicted w/ Prophet", linewidth=2)

plt.title("Actual vs Predicted values")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()
evaluate(y_test, forecast['yhat'][-len(test):])

## Modelling with Rolling window
In this section, we take the hyperparameters obtained from the previous step and apply them to a rolling-window forecasting setup. Our rolling window uses a training period of 8 years followed by a 2-year forecasting horizon.

In [ ]:
window_size = 96  # 8 years
forecast_horizon = 24  # 2-year ahead

preds = []
actuals = []
dates = []

lb_results = []
rmse_per_fold = []

for start in range(0, len(data) - window_size - forecast_horizon + 1, forecast_horizon):

    # Training and test windows
    train_window = data.iloc[start:start + window_size]
    test_window = data.iloc[start + window_size:start + window_size + forecast_horizon]

    # Initialize Prophet model with best params
    model = Prophet(
        changepoint_prior_scale=best_params['changepoint_prior_scale'],
        seasonality_prior_scale=best_params['seasonality_prior_scale'],
        seasonality_mode=best_params['seasonality_mode']
    )

    # Fit model
    model.fit(train_window)

    # Forecast one full horizon
    future = model.make_future_dataframe(periods=forecast_horizon, freq='MS')
    forecast = model.predict(future)

    # Extract forecast matching the test-window dates
    forecast_window = (
        forecast.set_index('ds')
                .loc[test_window['ds'].values]  # aligned on timestamp
    )

    # Store values
    preds.extend(forecast_window['yhat'].values)
    actuals.extend(test_window['y'].values)
    dates.extend(test_window['ds'].values)

    # Residuals for this window
    residuals = test_window['y'].values - forecast_window['yhat'].values

    # Ljung–Box at seasonal lag
    lb = acorr_ljungbox(
        residuals,
        lags=[12],
        return_df=True
    )

    lb_results.append({
        'ds': test_window['ds'].iloc[-1],   # end of forecast window
        'lb_stat': lb['lb_stat'].iloc[0],
        'p_value': lb['lb_pvalue'].iloc[0]
    })
    rmse = np.sqrt(mean_squared_error(test_window['y'].values, forecast_window['yhat'].values))
    rmse_per_fold.append(rmse)

rmse_mean = np.mean(rmse_per_fold)
rmse_std = np.std(rmse_per_fold, ddof=1)  # sample std

print(f"Mean RMSE: {rmse_mean:.3f}")
print(f"Std. Dev. RMSE: {rmse_std:.3f}")

#DataFrame for plotting
results_df = pd.DataFrame({
    'ds': dates,
    'actual': actuals,
    'predicted': preds
})
evaluate(actuals, preds)

In [ ]:
# Plot
plt.figure(figsize=(12, 6))
plt.plot(results_df["ds"], results_df["actual"], label="Actual", linewidth=2)
plt.plot(results_df["ds"], results_df["predicted"], label="Predicted w/ Prophet (Rolling Forecast)", linewidth=2)
plt.title("Actual vs Predicted values (Rolling Forecast)")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
residuals_r = np.array(actuals) - np.array(preds)

plt.figure(figsize=(12, 4))
plt.scatter(results_df["ds"], residuals_r)
plt.axhline(0)
plt.title("Residuals over time")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.show()

In [ ]:
lb_df = pd.DataFrame(lb_results)

plot_lb_test(lb_df, rolling_window=True)

## ACF computation and diagnosis
In this section we compute the ACF of the residuals to diagnose if the model is working well or not. First we used the regular model with no rolling window to then compare to our rolling window residuals.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Initialize Prophet model with best params
model = Prophet(
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    seasonality_mode=best_params['seasonality_mode']
)
model.fit(data)

# Cross-validated predictions (out-of-sample forecasts to avoid optimistic diagnostics.)
df_cv = cross_validation(
    model,
    horizon="365 days",
    period="30 days",
    initial="7304 days"
)

# Residuals
df_cv['residual'] = df_cv['y'] - df_cv['yhat']
residuals = df_cv['residual']

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

plot_acf(residuals, lags=24, ax=axes)
axes.set_title("ACF of Residuals")

plt.tight_layout()
plt.show()

Given the plot of the ACF, we can surmise these things:
* Most autocorrelation values lie within the confidence bounds;
* Most autocorrelation values fluctuate around zero;
* There are clear significant spikes at lags 12, 24, etc... for the ACF.

Based on the above, we conclude that the model successfully captures the overall trend and that there is no strong short-term autoregressive structure remaining in the residuals. However, the model does not fully capture the annual seasonal patterns, despite yearly seasonality being explicitly enabled. This behavior is likely due to the dataset showing cyclostationarity at lag 12.

Since Prophet is specifically designed to model seasonal effects directly, we avoid applying seasonal differencing at lag 12. Altering the dataset is not recommended for Prophet, as it would remove seasonal information that the model relies on for forecasting. Instead, we try to solve this issue by adjusting the model’s seasonal parameters to better capture the underlying annual cycle.


In [ ]:
model = Prophet(
    yearly_seasonality=False,
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    seasonality_mode=best_params['seasonality_mode']
)

model.add_seasonality(
    name='yearly',
    period=365.25,
    fourier_order=35  # increase flexibility
)

model.fit(data)

# Cross-validated predictions (out-of-sample forecasts to avoid optimistic diagnostics.)
df_cv = cross_validation(
    model,
    horizon="365 days",
    period="30 days",
    initial="7304 days"
)

# Residuals
residuals = df_cv['y'] - df_cv['yhat']

fig, axes = plt.subplots(1, 1, figsize=(12, 4))

plot_acf(residuals, lags=24, ax=axes)
axes.set_title("ACF of Residuals") 

plt.tight_layout()
plt.show()

Increasing the Fourier order of the yearly seasonal component does not eliminate the residual autocorrelation at lag 12. This indicates the presence of time-varying annual seasonality (cyclostationarity), which cannot be fully captured by Prophet’s global seasonal formulation (Prophet assumes seasonality is globally consistent overtime). Nevertheless, the model adequately captures the overall trend and short-term dynamics, and the rolling-window forecasting strategy helps mitigate the impact of this remaining seasonal structure.

In [ ]:
# Using rolling forecast residuals
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

plot_acf(residuals_r, lags=24, ax=axes)
axes.set_title("ACF of Residuals")

plt.tight_layout()
plt.show()

In [ ]:
# Using most recent rolling forecast residuals
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

plot_acf(residuals_r[-forecast_horizon:], lags=12, ax=axes)
axes.set_title("ACF of Residuals")

plt.tight_layout()
plt.show()
print(lb_df.iloc[-1])

# Traditional Machine Learning 

In this section, we attempted to use ML models to predict 'tdiff'. Contrarily to the other sections, we explored calculating tdiff by subtracting the predictions between tmin and tmax. 

Additionally, due to the nature of the models, the predictions had to be made iteratively, having its features changed for every predicted month according to the previous prediction. This means that if we wanted to predict for February of 2019, we would need to take into account the predicted values of January of the same year.

Details and discussion about the results will be presented in the report.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

window_size = 96  # 8 years
forecast_horizon = 24  # 2-year ahead

In [ ]:
"""
This function performs iterative forecasting
The feature description is described in the cell below
"""
def recursive_forecast(model, data, n_steps=12, n_lags=12):
    last_data = data.copy()
    forecast = []
    
    for i in range(n_steps):
        lags = [last_data['y'].iloc[-lag] for lag in range(1, n_lags + 1)]
        
        rolling_3 = np.mean([last_data['y'].iloc[-j] for j in range(1, min(4, len(last_data)+1))])
        rolling_6 = np.mean([last_data['y'].iloc[-j] for j in range(1, min(7, len(last_data)+1))])
        
        next_month = last_data.index[-1] + pd.DateOffset(months=1)
        month_sin = np.sin(2 * np.pi * next_month.month / 12)
        month_cos = np.cos(2 * np.pi * next_month.month / 12)
        
        X_next = np.array(lags + [rolling_3, rolling_6, next_month.month, month_sin, month_cos]).reshape(1, -1)
        
        y_next = model.predict(X_next)[0]
        forecast.append(y_next)
        
        last_data = pd.concat([last_data, pd.DataFrame({'y': [y_next]}, index=[next_month])])
    
    forecast_index = pd.date_range(start=last_data.index[-n_steps], periods=n_steps, freq='MS')
    return pd.Series(forecast, index=forecast_index)

def get_model_prediction(model, train, steps):
    X_train = train.drop(columns=['y'])
    y_train = train['y']

    model.fit(X_train, y_train)
    return recursive_forecast(model, train, n_steps=steps)

In [ ]:
"""
This function prepares the data creating the following features:
- lag_1 to lag_12: previous 12 months of the target variable
- rolling_3: 3-month rolling mean of the target variable
- rolling_6: 6-month rolling mean of the target variable
- month: month of the year as an integer (1-12)
- month_sin: sine transformation of the month
- month_cos: cosine transformation of the month
"""
global_data = load_data()
def get_data(target):
    loaded_data = global_data
    data = pd.DataFrame({
        'date': loaded_data['date'],
        'y': loaded_data[target]
    })
    
    data.set_index('date', inplace=True)
    
    for lag in range(1, 13):
        data[f'lag_{lag}'] = data['y'].shift(lag)
    
    data['rolling_3'] = data['y'].shift(1).rolling(3).mean()
    data['rolling_6'] = data['y'].shift(1).rolling(6).mean()
    
    data['month'] = data.index.month
    data['month_sin'] = np.sin(2 * np.pi * data['month']/12)
    data['month_cos'] = np.cos(2 * np.pi * data['month']/12)
    
    data.dropna(inplace=True)

    return data

## Random Forest

In [ ]:
"""
This function performs sliding window cross-validation using Random Forest
using given hyperparameters. This was done to facilitate grid search.

Instead of predicting directly tdiff, we predict tmax and tmin separately.
"""
def sliding_window_rf(params):
    preds = []
    actuals = []
    dates = []
    lb_results = []
    rmse_per_fold = []

    max_data = get_data('tmax')
    min_data = get_data('tmin')
    diff_data = get_data('tdiff')

    for start in range(0, len(diff_data) - window_size - forecast_horizon + 1, forecast_horizon):
        max_train_window = max_data.iloc[start:start + window_size]
        min_train_window = min_data.iloc[start:start + window_size]
        diff_test_window = diff_data.iloc[start + window_size:start + window_size + forecast_horizon]
        
        rf_max = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )
        
        rf_min = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        max_pred_rf = get_model_prediction(rf_max, max_train_window, forecast_horizon)
        min_pred_rf = get_model_prediction(rf_min, min_train_window, forecast_horizon)
        diff_pred_rf = max_pred_rf - min_pred_rf

        preds.extend(diff_pred_rf.values)
        actuals.extend(diff_test_window['y'].values)
        dates.extend(diff_test_window.index)

        residuals = diff_test_window['y'].values - diff_pred_rf.values
        rmse = np.sqrt(mean_squared_error(diff_test_window['y'].values, diff_pred_rf.values))
        rmse_per_fold.append(rmse)

        lb = acorr_ljungbox(
            residuals,
            lags=[12],
            return_df=True
        )

        lb_results.append({
            'ds': diff_test_window.index[-1], 
            'lb_stat': lb['lb_stat'].iloc[0],
            'p_value': lb['lb_pvalue'].iloc[0]
        })

    rmse_mean = np.mean(rmse_per_fold)
    rmse_std = np.std(rmse_per_fold, ddof=1) 

    results_df = pd.DataFrame({
        'ds': dates,
        'actual': actuals,
        'predicted': preds
    })

    return -rmse_mean, rmse_std, results_df, lb_results

In [ ]:
"""
Grid Search for Random Forest hyperparameters
"""
param_grid_rf = {
    "n_estimators": [200, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5]
}

best_score = float("-inf")
best_params_rf = None
results = []
results_df = pd.DataFrame()
results_lb = []

keys, values = zip(*param_grid_rf.items())

for combo in itertools.product(*values):
    params = dict(zip(keys, combo))

    score, std, new_results_df, new_results_lb = sliding_window_rf(params)

    results.append((params, score))

    if score > best_score:
        best_score = score
        best_params_rf = params
        results_df = new_results_df
        results_lb = new_results_lb

print(f"Params: {best_params_rf} → Score: {-best_score:.4f}")
plots_from_ml_results(results_df, results_lb)

In [ ]:
"""
Execution with best Random Forest parameters to predict the last two years
"""
forecast_horizon = 24  # 2-year ahead

preds = []
actuals = []
dates = []
lb_results = []

max_data = get_data('tmax')
min_data = get_data('tmin')
diff_data = get_data('tdiff')

train_max = max_data.iloc[:-forecast_horizon]
train_min = min_data.iloc[:-forecast_horizon]
test_diff = diff_data.iloc[-forecast_horizon:]

rf_max = RandomForestRegressor(
    **best_params_rf,
    random_state=42,
    n_jobs=-1
)
        

rf_min = RandomForestRegressor(
    **best_params_rf,
    random_state=42,
    n_jobs=-1
)
      

max_pred_rf = get_model_prediction(rf_max, train_max, forecast_horizon)
min_pred_rf = get_model_prediction(rf_min, train_min, forecast_horizon)
diff_pred_rf = max_pred_rf - min_pred_rf

preds.extend(diff_pred_rf.values)
actuals.extend(test_diff['y'].values)
dates.extend(test_diff.index)

residuals = test_diff['y'].values - diff_pred_rf.values

lb = acorr_ljungbox(
    residuals,
    lags=[12],
    return_df=True
)

lb_results.append({
    'ds': test_diff.index[-1],   
    'lb_stat': lb['lb_stat'].iloc[0],
    'p_value': lb['lb_pvalue'].iloc[0]
})

evaluate(actuals, preds)
results_df = pd.DataFrame({
    'ds': dates,
    'actual': actuals,
    'predicted': preds
})

plots_from_ml_results(results_df, lb_results)

## XGBoost

In [ ]:
"""
This function performs sliding window cross-validation using XGBoost
using given hyperparameters. This was done to facilitate grid search.

Instead of predicting directly tdiff, we predict tmax and tmin separately.
"""
def sliding_window_xgb(params):
    preds = []
    actuals = []
    dates = []
    lb_results = []
    rmse_per_fold = []

    max_data = get_data('tmax')
    min_data = get_data('tmin')
    diff_data = get_data('tdiff')

    for start in range(0, len(diff_data) - window_size - forecast_horizon + 1, forecast_horizon):
        max_train_window = max_data.iloc[start:start + window_size]
        min_train_window = min_data.iloc[start:start + window_size]
        diff_test_window = diff_data.iloc[start + window_size:start + window_size + forecast_horizon]

        
        xgb_max = XGBRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )
        
        xgb_min = XGBRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        max_pred_xgb = get_model_prediction(xgb_max, max_train_window, forecast_horizon)
        min_pred_xgb = get_model_prediction(xgb_min, min_train_window, forecast_horizon)
        diff_pred_xgb = max_pred_xgb - min_pred_xgb

        preds.extend(diff_pred_xgb.values)
        actuals.extend(diff_test_window['y'].values)
        dates.extend(diff_test_window.index)

        residuals = diff_test_window['y'].values - diff_pred_xgb.values
        rmse = np.sqrt(mean_squared_error(diff_test_window['y'].values, diff_pred_xgb.values))
        rmse_per_fold.append(rmse)

        lb = acorr_ljungbox(
            residuals,
            lags=[12],
            return_df=True
        )

        lb_results.append({
            'ds': diff_test_window.index[-1],
            'lb_stat': lb['lb_stat'].iloc[0],
            'p_value': lb['lb_pvalue'].iloc[0]
        })

    rmse_mean = np.mean(rmse_per_fold)
    rmse_std = np.std(rmse_per_fold, ddof=1)

    results_df = pd.DataFrame({
        'ds': dates,
        'actual': actuals,
        'predicted': preds
    })

    return -rmse_mean, rmse_std, results_df, lb_results

In [ ]:
"""
Grid Search for XGBoost hyperparameters
"""
param_grid_xgb = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9]
}

best_score = float("-inf")
best_params_xgb = None
results = []
results_df = pd.DataFrame()
results_lb = []

keys, values = zip(*param_grid_xgb.items())

for combo in itertools.product(*values):
    params = dict(zip(keys, combo))

    score, std, new_results_df, new_results_lb = sliding_window_xgb(params)

    results.append((params, score))

    if score > best_score:
        best_score = score
        best_params_xgb = params
        results_df = new_results_df
        results_lb = new_results_lb

print(f"Params: {best_params_xgb} → Score: {-best_score:.4f}")
plots_from_ml_results(results_df, results_lb)

In [ ]:
"""
Execution with best XGBoost parameters to predict the last two years
"""
forecast_horizon = 24  # 2-year ahead

preds = []
actuals = []
dates = []
lb_results = []

max_data = get_data('tmax')
min_data = get_data('tmin')
diff_data = get_data('tdiff')

train_max = max_data.iloc[:-forecast_horizon]
train_min = min_data.iloc[:-forecast_horizon]
test_diff = diff_data.iloc[-forecast_horizon:]

xgb_max = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1
)

xgb_min = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1
)

max_pred_xgb = get_model_prediction(xgb_max, train_max, forecast_horizon)
min_pred_xgb = get_model_prediction(xgb_min, train_min, forecast_horizon)
diff_pred_xgb = max_pred_xgb - min_pred_xgb

preds.extend(diff_pred_xgb.values)
actuals.extend(test_diff['y'].values)
dates.extend(test_diff.index)

residuals = test_diff['y'].values - diff_pred_xgb.values
lb = acorr_ljungbox(
    residuals,
    lags=[12],
    return_df=True
)

lb_results.append({
    'ds': test_diff.index[-1],  
    'lb_stat': lb['lb_stat'].iloc[0],
    'p_value': lb['lb_pvalue'].iloc[0]
})

evaluate(actuals, preds)
results_df = pd.DataFrame({
    'ds': dates,
    'actual': actuals,
    'predicted': preds
})

plots_from_ml_results(results_df, lb_results)